In [8]:
import imageio
from tqdm import tqdm
import os
from os import path as osp
import torch

import imageio
from PIL import Image
import torchvision.transforms as transforms


import torchvision.transforms as TT
import torch.distributed as dist
import sys
sys.path.append('/home/user/ttt-video-dit_v2')
from ttt.models.configs import VaeModelConfig
from ttt.models.vae.autoencoder import VideoAutoencoderInferenceWrapper
from ttt.infra.parallelisms import init_distributed
from ttt.infra.config_manager import JobConfig
from typing import List

In [9]:
def pad_video(frames: List[torch.Tensor], target_num_frames: int) -> List[torch.Tensor]:
    """Pad video frames to reach target number of frames by repeating the last frame.
    
    Args:
        frames: List of video frames as tensors
        target_num_frames: Target number of frames to pad to
        
    Returns:
        Padded list of frames
    """
    pad_num = target_num_frames - len(frames)
    return frames + [frames[-1]] * pad_num

def crop_video(frames: List[torch.Tensor], target_num_frames: int) -> List[torch.Tensor]:
    """Crop video frames to target number of frames by removing frames from the middle.
    
    Args:
        frames: List of video frames as tensors
        target_num_frames: Target number of frames to crop to
        
    Returns:
        Cropped list of frames
    """
    overflow = len(frames) - target_num_frames
    start = overflow // 2
    return frames[start : start + target_num_frames]

def get_vae(
    vae_weight_path: str,
    fps: int = 16,
    tiling_window_unit: int = 1,
    device: str = "cuda"
) -> VideoAutoencoderInferenceWrapper:
    """Initialize and return a VAE model.
    
    Args:
        vae_weight_path: Path to VAE weights
        fps: Frames per second
        tiling_window_unit: Temporal tiling window unit
        device: Device to load model on
        
    Returns:
        Initialized VAE model
    """
   
    vae = VideoAutoencoderInferenceWrapper(
        vae_weight_path,
        VaeModelConfig.get_encoder_config(temporal_tiling_window=fps*tiling_window_unit),
        VaeModelConfig.get_decoder_config(temporal_tiling_window=2),
    ).to(device).to(torch.bfloat16)
    
    vae.eval()
    print(f"VAE initialized: (enc tiling window = {vae.encoder_temporal_tiling_window}, "
          f"dec tiling window = {vae.decoder_temporal_tiling_window})")
    
    return vae



def resample_video_to_fps(video_path, target_fps=16, target_size=(640, 480), video_lenth=3):
    # Load video
    video_reader = imageio.get_reader(video_path, "ffmpeg")
    meta = video_reader.get_meta_data()
    original_fps = meta["fps"]
    duration = meta["duration"]
    total_frames = int(video_lenth * target_fps)
    #target_num_frames = target_fps * video_length
    print("total_frames====================.  ",total_frames)

    # Load all frames
    original_frames = [Image.fromarray(f) for f in video_reader]
    n_original = len(original_frames)

    # Resize and transform
    resize = transforms.Resize(target_size)
    to_tensor = transforms.ToTensor()

    # Resample frames to target_fps
    resampled_frames = []
    for i in range(total_frames):
        # Map each target frame index to original frame index
        orig_index = int(i * (n_original / total_frames))
        orig_index = min(orig_index, n_original - 1)  # Clamp
        img = resize(original_frames[orig_index])
        tensor = to_tensor(img)
        resampled_frames.append(tensor)
    

    return resampled_frames  # List of [C, H, W] tensors


def load_and_process_video(video_path, target_fps=30, target_size=(640, 480)):
    video_reader = imageio.get_reader(video_path, "ffmpeg")
    original_fps = video_reader.get_meta_data()["fps"]

    # Optional: adjust frame sampling rate if FPS is different
    frame_step = int(round(original_fps / target_fps)) if target_fps < original_fps else 1

    resize = transforms.Resize(target_size)
    to_tensor = transforms.ToTensor()

    frames = []
    for i, frame in enumerate(video_reader):
        if i % frame_step != 0:
            continue
        img = Image.fromarray(frame)
        img_resized = resize(img)
        tensor = to_tensor(img_resized)
        frames.append(tensor)

    return frames  # List of tensors

In [24]:
episode_dir = "../../ttt_video_dataset/data/clips"
FPS = 16
video_length = 3
tiling_window_unit = 1
batch_size = 1
vae_weight_path = "../../ttt-video-dit_v1/CogVideoX-2b-sat/vae/3d-vae.pt"
output_dir = "../../ttt_video_dataset/data/encoded_v_9/"

In [25]:
#TARGET_NUM_FRAMES = FPS * video_length + 1
TARGET_NUM_FRAMES = FPS * video_length
print(f'Precomputing {video_length}s video embeddings\n\tfrom: {episode_dir}\n\tto: {output_dir}.')
print(f'.mp4 files should have {TARGET_NUM_FRAMES} frames.')

# Initialize VAE
local_rank = int(os.environ.get("LOCAL_RANK", 0))

Precomputing 3s video embeddings
	from: ../../ttt_video_dataset/data/clips
	to: ../../ttt_video_dataset/data/encoded_v_9/.
.mp4 files should have 48 frames.


In [26]:
vae = get_vae(
        vae_weight_path=vae_weight_path,
        fps=FPS,
        tiling_window_unit=tiling_window_unit,
        device=f"cuda:{local_rank}"
    )
assert vae.encoder_temporal_tiling_window == FPS*tiling_window_unit

Working with z of shape (1, 16, 32, 32) = 16384 dimensions.
Missing keys:  []
Unexpected keys:  ['loss.logvar', 'loss.perceptual_loss.scaling_layer.shift', 'loss.perceptual_loss.scaling_layer.scale', 'loss.perceptual_loss.net.slice1.0.weight', 'loss.perceptual_loss.net.slice1.0.bias', 'loss.perceptual_loss.net.slice1.2.weight', 'loss.perceptual_loss.net.slice1.2.bias', 'loss.perceptual_loss.net.slice2.5.weight', 'loss.perceptual_loss.net.slice2.5.bias', 'loss.perceptual_loss.net.slice2.7.weight', 'loss.perceptual_loss.net.slice2.7.bias', 'loss.perceptual_loss.net.slice3.10.weight', 'loss.perceptual_loss.net.slice3.10.bias', 'loss.perceptual_loss.net.slice3.12.weight', 'loss.perceptual_loss.net.slice3.12.bias', 'loss.perceptual_loss.net.slice3.14.weight', 'loss.perceptual_loss.net.slice3.14.bias', 'loss.perceptual_loss.net.slice4.17.weight', 'loss.perceptual_loss.net.slice4.17.bias', 'loss.perceptual_loss.net.slice4.19.weight', 'loss.perceptual_loss.net.slice4.19.bias', 'loss.perceptual